# Stage 11 - Final Report: Plots, Tables, and Summary

Generates all publication-quality figures (P1-P24) and summary tables (T1-T14)
for the AFML NVDA project report.  Loads all intermediate outputs from Stages 0-9.

Reference: Lopez de Prado, *Advances in Financial Machine Learning*, Wiley (2018)

In [ ]:
import os
import sys, warnings, json
sys.path.insert(0, os.path.abspath('../..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

FIGURES = '../../reports/figures'
TABLES  = '../../reports'
plt.rcParams.update({'font.size': 11, 'grid.alpha': 0.3, 'figure.dpi': 120})
STYLE = dict(dpi=150, bbox_inches='tight')
print('Imports OK')

## Load all artifacts

In [ ]:
# Stage 0-1
clean  = pd.read_parquet('../../data/processed/legacy/nvda_clean.parquet')
dbar   = pd.read_parquet('../../data/processed/legacy/nvda_dollar_bars.parquet')
cusum  = pd.read_parquet('../../data/processed/legacy/nvda_cusum_events.parquet')

# Stage 2
labels  = pd.read_parquet('../../data/processed/legacy/nvda_labels.parquet')
weights = pd.read_parquet('../../data/processed/legacy/nvda_sample_weights.parquet')

# Stage 3
fracdiff = pd.read_parquet('../../data/processed/legacy/nvda_fracdiff.parquet')
features = pd.read_parquet('../../data/processed/legacy/nvda_features.parquet')

# Stage 4-6
mod  = pd.read_parquet('../../data/processed/pooled_modelling.parquet')
cv_res = pd.read_parquet('../../data/processed/cv_results.parquet')
tlog = pd.read_parquet('../../data/processed/tuning_log.parquet')
fimp = pd.read_parquet('../../data/processed/feature_importance.parquet')
with open('../../models/best_params.json') as f: bparams = json.load(f)

# Stage 7-8
oos_pred = pd.read_parquet('../../data/processed/oos_predictions_pooled.parquet')
meta_lab = pd.read_parquet('../../data/processed/meta_labels_pooled.parquet')
meta_pred= pd.read_parquet('../../data/processed/meta_oos_predictions_pooled.parquet')
positions= pd.read_parquet('../../data/processed/bet_sizes_pooled.parquet')
bt_res   = pd.read_parquet('../../data/processed/backtest_results.parquet')
bt_stats = pd.read_csv('../../data/processed/backtest_stats.csv', index_col=0)
cpcv_res = pd.read_parquet('../../data/processed/cpcv_results.parquet')

# Stage 9 (structural breaks) - may not exist yet
import os
if os.path.exists('../../data/processed/legacy/nvda_bsadf.parquet'):
    bsadf = pd.read_parquet('../../data/processed/legacy/nvda_bsadf.parquet')['bsadf']
    print('BSADF loaded')
else:
    bsadf = None
    print('BSADF not found - run notebook 12 first')

print('All artifacts loaded.')
print(f'Clean: {clean.shape}  Labels: {labels.shape}  Modelling: {mod.shape}')

## T1 — Dataset Summary Statistics

In [ ]:
log_ret = np.log(clean['Adj Close'] / clean['Adj Close'].shift(1)).dropna()
jb_stat, jb_p = stats.jarque_bera(log_ret)
outliers = (log_ret.abs() > 5 * log_ret.std()).sum()

T1 = pd.Series({
    'Asset':           'NVIDIA Corporation (NVDA)',
    'Frequency':       'Daily (trading days)',
    'Observations':    len(clean),
    'Date range':      f"{clean.index[0].date()} to {clean.index[-1].date()}",
    'Adj Close min':   f"${clean['Adj Close'].min():.3f}",
    'Adj Close max':   f"${clean['Adj Close'].max():.2f}",
    'Mean daily log return': f"{log_ret.mean():.5f}",
    'Std daily log return':  f"{log_ret.std():.5f}",
    'Skewness':        f"{log_ret.skew():.4f}",
    'Excess kurtosis': f"{log_ret.kurtosis():.4f}",
    'Jarque-Bera stat':f"{jb_stat:.2f}",
    'JB p-value':      f"{jb_p:.2e}",
    'Outlier days (>5sigma)': outliers,
    'Missing values':  0,
}, name='T1: Dataset Summary')
print(T1.to_string())
T1.to_csv(f'{TABLES}/T1_dataset_summary.csv')

## P1 — NVDA Adj Close price chart (log scale, 2005-2025)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), dpi=120)
ax.semilogy(clean.index, clean['Adj Close'], color='steelblue', lw=0.8, label='Adj Close')
ax.set_title('NVDA Adjusted Close Price 2005-2025  (AFML Stage 0)', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Adj Close (log scale, USD)')
ax.legend(fontsize=10); ax.grid(True)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(f'{FIGURES}/P1_nvda_price_chart.png', **STYLE)
plt.show(); print('Saved P1')

## P2 — Daily log return distribution + QQ plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=120)
# Histogram
ax = axes[0]
ax.hist(log_ret, bins=80, color='steelblue', alpha=0.7, edgecolor='white', density=True)
xs = np.linspace(log_ret.min(), log_ret.max(), 200)
ax.plot(xs, stats.norm.pdf(xs, log_ret.mean(), log_ret.std()), 'r-', lw=1.5, label='Normal fit')
ax.set_title('Daily Log Return Distribution', fontweight='bold')
ax.set_xlabel('Log return'); ax.set_ylabel('Density')
jb_s, jb_p2 = stats.jarque_bera(log_ret)
ax.text(0.98, 0.95, f'JB={jb_s:.0f}\np={jb_p2:.2e}', transform=ax.transAxes,
        ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.legend(fontsize=10); ax.grid(True)
# QQ
ax = axes[1]
stats.probplot(log_ret, dist='norm', plot=ax)
ax.set_title('QQ Plot vs Normal', fontweight='bold')
ax.grid(True)
plt.tight_layout()
plt.savefig(f'{FIGURES}/P2_returns_distribution.png', **STYLE)
plt.show(); print('Saved P2')

## T2 & P3 — Dollar Bar Calibration

In [ ]:
# Count bars per year
dbar['year'] = dbar.index.year
dollar_bars_per_year = dbar.groupby('year').size()
clean['year'] = clean.index.year
time_bars_per_year  = clean.groupby('year').size()

T2 = pd.DataFrame({
    'Time bars (daily)': time_bars_per_year,
    'Dollar bars':       dollar_bars_per_year,
}).dropna().astype(int)
T2['Dollar/Time ratio'] = (T2['Dollar bars'] / T2['Time bars (daily)']).round(3)
print('T2: Dollar bar counts per year')
print(T2.to_string())
T2.to_csv(f'{TABLES}/T2_dollar_bar_calibration.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), dpi=120)
yrs = T2.index
w = 0.4
ax.bar(yrs - w/2, T2['Time bars (daily)'], width=w, label='Time bars (daily)', color='steelblue', alpha=0.8)
ax.bar(yrs + w/2, T2['Dollar bars'],       width=w, label='Dollar bars',       color='darkorange', alpha=0.8)
ax.set_title('Bar Count per Year: Time Bars vs Dollar Bars  (AFML Stage 1)', fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Number of bars')
ax.legend(fontsize=10); ax.grid(axis='y')
plt.tight_layout()
plt.savefig(f'{FIGURES}/P3_bars_per_year.png', **STYLE)
plt.show(); print('Saved P3')

## T3 & P4 — CUSUM Events

In [ ]:
T3 = pd.Series({
    'Total CUSUM events': len(cusum),
    'Date range': f'{cusum.index[0].date()} to {cusum.index[-1].date()}',
    'Avg inter-event gap (days)': round(cusum.index.to_series().diff().dt.days.mean(), 1),
    'Median inter-event gap (days)': round(cusum.index.to_series().diff().dt.days.median(), 1),
    'Events per year': round(len(cusum) / ((cusum.index[-1] - cusum.index[0]).days / 365.25), 1),
}, name='T3: CUSUM Filter')
print(T3.to_string())
T3.to_csv(f'{TABLES}/T3_cusum_calibration.csv')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), dpi=120)
ax.semilogy(clean.index, clean['Adj Close'], color='steelblue', lw=0.6, alpha=0.8, label='Adj Close')
event_prices = clean.loc[cusum.index, 'Adj Close']
ax.scatter(event_prices.index, event_prices.values, color='red', s=12, zorder=5,
           alpha=0.6, label=f'CUSUM events (n={len(cusum)})')
ax.set_title('NVDA Price with CUSUM Events  (AFML Stage 1)', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Adj Close (log scale)')
ax.legend(fontsize=10); ax.grid(True)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(f'{FIGURES}/P4_cusum_events.png', **STYLE)
plt.show(); print('Saved P4')

## T4, P5, P6 — Labels (Triple-Barrier)

In [ ]:
label_counts = labels['bin'].value_counts().sort_index()
T4 = pd.DataFrame({
    'Count': label_counts,
    'Fraction': (label_counts / len(labels)).round(4),
})
T4.index = ['Sell (-1)', 'Buy (+1)'] if -1 in T4.index else T4.index
print('T4: Label distribution')
print(T4.to_string())
T4.to_csv(f'{TABLES}/T4_label_distribution.csv')

In [ ]:
# P5 - events on price coloured by label
fig, ax = plt.subplots(figsize=(14, 5), dpi=120)
ax.semilogy(clean.index, clean['Adj Close'], color='lightgrey', lw=0.6)
color_map = {1.0: 'green', -1.0: 'red'}
label_name = {1.0: 'Buy (+1)', -1.0: 'Sell (-1)'}
for lbl, grp in labels.groupby('bin'):
    prices = clean.loc[grp.index, 'Adj Close']
    ax.scatter(prices.index, prices.values, color=color_map.get(lbl,'grey'),
               s=20, zorder=5, alpha=0.7, label=label_name.get(lbl, str(lbl)))
ax.set_title('Triple-Barrier Events Coloured by Label  (AFML Stage 2)', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Adj Close (log scale)')
ax.legend(fontsize=10); ax.grid(True)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(f'{FIGURES}/P5_label_events.png', **STYLE)
plt.show(); print('Saved P5')

In [ ]:
# P6 - label distribution bar chart
fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
bars = ax.bar(['-1 (Sell)', '+1 (Buy)'],
             [label_counts.get(-1.0, 0), label_counts.get(1.0, 0)],
             color=['tomato', 'seagreen'], edgecolor='white', alpha=0.85)
for bar, count in zip(bars, [label_counts.get(-1.0, 0), label_counts.get(1.0, 0)]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(count), ha='center', va='bottom', fontweight='bold')
ax.set_title('Triple-Barrier Label Distribution  (AFML Stage 2)', fontweight='bold')
ax.set_xlabel('Label'); ax.set_ylabel('Count')
ax.set_ylim(0, max(label_counts.values) * 1.15)
ax.grid(axis='y')
plt.tight_layout()
plt.savefig(f'{FIGURES}/P6_label_distribution.png', **STYLE)
plt.show(); print('Saved P6')

## T5, P7, P8 — Sample Weights

In [ ]:
T5 = weights['weight'].describe().rename('Sample weights').round(4)
print('T5: Sample weight statistics')
print(T5.to_string())
T5.to_csv(f'{TABLES}/T5_sample_weight_stats.csv')

In [ ]:
# P7 - concurrency (proxy: weight inversely reflects uniqueness)
fig, ax = plt.subplots(figsize=(14, 4), dpi=120)
ax.plot(weights.index, weights['weight'], color='purple', lw=0.8, alpha=0.8)
ax.axhline(weights['weight'].mean(), color='red', ls='--', lw=1, label=f'Mean={weights["weight"].mean():.3f}')
ax.set_title('Sample Weights over Time  (AFML Stage 2 - uniqueness x return x time decay)', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Sample weight')
ax.legend(fontsize=10); ax.grid(True)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(f'{FIGURES}/P7_sample_weights_time.png', **STYLE)
plt.show(); print('Saved P7')

In [ ]:
# P8 - weight histogram
fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
ax.hist(weights['weight'], bins=30, color='purple', alpha=0.7, edgecolor='white')
ax.axvline(weights['weight'].mean(), color='red', ls='--', lw=1.5,
           label=f'Mean={weights["weight"].mean():.3f}')
ax.set_title('Sample Weight Distribution  (AFML Stage 2)', fontweight='bold')
ax.set_xlabel('Weight'); ax.set_ylabel('Count')
ax.legend(fontsize=10); ax.grid(True)
plt.tight_layout()
plt.savefig(f'{FIGURES}/P8_weight_histogram.png', **STYLE)
plt.show(); print('Saved P8')

## T6 — Fracdiff sweep results (ADF p-value and correlation)

In [ ]:
# Reload fracdiff and show d* result
fd_star = fracdiff['fracdiff_d_star']
from src.fracdiff import find_min_d, frac_diff_ffd
from statsmodels.tsa.stattools import adfuller

log_close = np.log(clean['Adj Close'])
d_values = np.arange(0, 1.05, 0.05)
rows = []
for d in d_values:
    series = frac_diff_ffd(log_close, d=d, threshold=1e-5).dropna()
    if len(series) < 50:
        rows.append({'d': round(d,2), 'ADF_stat': np.nan, 'p_value': np.nan, 'corr': np.nan, 'n_obs': len(series)})
        continue
    adf_result = adfuller(series, maxlag=1, regression='c', autolag=None)
    aligned = log_close.reindex(series.index)
    corr = series.corr(aligned)
    rows.append({'d': round(d,2), 'ADF_stat': round(adf_result[0],4), 'p_value': round(adf_result[1],4),
                 'corr_with_log_price': round(corr,4), 'n_obs': len(series)})

T6 = pd.DataFrame(rows).set_index('d')
T6['stationary'] = T6['p_value'] < 0.05
print('T6: Fractional differentiation sweep (d* criterion: smallest d with p<0.05 and corr>0.9)')
print(T6.to_string())
T6.to_csv(f'{TABLES}/T6_fracdiff_sweep.csv')

## T7 — Feature list with descriptions

In [ ]:
feature_descriptions = {
    'ret_5d':              '5-day rolling log return',
    'ret_10d':             '10-day rolling log return',
    'ret_20d':             '20-day rolling log return (medium-horizon momentum)',
    'ret_60d':             '60-day rolling log return',
    'momentum_12_1':       '252-day minus 21-day log return (12-month momentum)',
    'rsi_14':              '14-day RSI using Wilders smoothing',
    'vol_20d':             '20-day rolling volatility (annualised)',
    'vol_50d':             '50-day rolling volatility (annualised)',
    'log_dollar_volume':   'log(Adj Close x Volume) — trading activity proxy',
    'volume_ratio':        'Volume / 20-day average volume — abnormal activity',
    'corwin_schultz_spread': 'CS bid-ask spread from H/L (daily OHLCV approx.)',
    'bekker_parkinson_vol':  'Range-based daily variance from H/L',
    'amihud_illiquidity':  '|return| / dollar volume, 20-day mean — price impact',
    'roll_spread':         'Roll (1984) spread from return serial covariance',
    'shannon_entropy':     '50-day rolling Shannon entropy of discretised returns',
    'lempel_ziv_complexity': '50-day rolling normalised LZ-76 complexity',
    'fracdiff':            'FFD log-price at optimal d*=0.25 (ADF p=0.012, corr=0.916)',
}
T7 = pd.DataFrame.from_dict(feature_descriptions, orient='index', columns=['Description'])
T7.index.name = 'Feature'
print('T7: Feature list with descriptions')
print(T7.to_string())
T7.to_csv(f'{TABLES}/T7_feature_list.csv')

## T8 & P12 & P13 — Model CV Performance

In [ ]:
# T8: CV accuracy comparison
T8 = pd.DataFrame({
    'Model': ['Majority baseline', 'RF (untuned)', 'XGB (untuned)', 'RF (tuned)', 'XGB (tuned)', 'RF (final, 15 feat)'],
    'Mean weighted CV accuracy': [0.5846, cv_res['RF'].mean(), cv_res['XGB'].mean(),
                                  bparams['rf']['mean_score'], bparams['xgb']['mean_score'],
                                  0.6411],
    'Std': [None, cv_res['RF'].std(), cv_res['XGB'].std(),
            bparams['rf']['std_score'], bparams['xgb']['std_score'], 0.093],
    'Beats baseline': ['—', False, False, True, True, True],
})
print('T8: CV Accuracy Comparison')
print(T8.to_string(index=False))
T8.to_csv(f'{TABLES}/T8_cv_accuracy.csv', index=False)

In [ ]:
# P12 - RF confusion matrix (using OOS predictions)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
y_true = mod['label'].values
y_pred = oos_pred['pred_class'].values
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4), dpi=120)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['-1', '+1'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('OOS Primary Model Confusion Matrix  (AFML Stage 4)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES}/P12_rf_confusion_matrix.png', **STYLE)
plt.show(); print('Saved P12')

In [ ]:
# P13 - Purged CV scores per fold
fold_rows = cv_res[cv_res.index.str.startswith('fold_')]
n_folds = len(fold_rows)
folds = [f'fold_{i}' for i in range(1, n_folds + 1)]
x = np.arange(n_folds)
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
ax.bar(x - w/2, fold_rows['RF'].values, width=w, label='RF', color='steelblue', alpha=0.85)
ax.bar(x + w/2, fold_rows['XGB'].values, width=w, label='XGB', color='darkorange', alpha=0.85)
ax.axhline(0.5846, color='red', ls='--', lw=1.2, label='Majority baseline (0.5846)')
ax.axhline(fold_rows['RF'].mean(), color='steelblue', ls=':', lw=1)
ax.axhline(fold_rows['XGB'].mean(), color='darkorange', ls=':', lw=1)
ax.set_xticks(x); ax.set_xticklabels([f'Fold {i}' for i in range(1, n_folds + 1)])
ax.set_title('Purged K-Fold CV Accuracy per Fold  (AFML Stage 4)', fontweight='bold')
ax.set_ylabel('Weighted accuracy'); ax.set_ylim(0.3, 0.85)
ax.legend(fontsize=10); ax.grid(axis='y')
plt.tight_layout()
plt.savefig(f'{FIGURES}/P13_purged_cv_scores.png', **STYLE)
plt.show(); print('Saved P13')

## T9 & P14 — Hyperparameter Tuning

In [ ]:
T9 = pd.DataFrame([
    {'Model': 'RF', 'n_estimators': bparams['rf']['params']['n_estimators'],
     'max_depth': bparams['rf']['params']['max_depth'],
     'min_samples_leaf': bparams['rf']['params']['min_samples_leaf'],
     'max_features': bparams['rf']['params']['max_features'],
     'Best CV': round(bparams['rf']['mean_score'],4),
     'DSR_CV': round(bparams['rf']['dsr_cv_trials']['dsr'],4)},
    {'Model': 'XGB', 'n_estimators': bparams['xgb']['params']['n_estimators'],
     'max_depth': bparams['xgb']['params']['max_depth'],
     'min_samples_leaf': bparams['xgb']['params'].get('min_samples_leaf','—'),
     'max_features': '—',
     'Best CV': round(bparams['xgb']['mean_score'],4),
     'DSR_CV': round(bparams['xgb']['dsr_cv_trials']['dsr'],4)},
])
print('T9: Best Hyperparameters')
print(T9.to_string(index=False))
T9.to_csv(f'{TABLES}/T9_best_hyperparams.csv', index=False)

In [ ]:
# P14 - hyperparameter search landscape (mean_score vs max_depth, coloured by model)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=120)
for ax, model in zip(axes, ['RF', 'XGB']):
    sub = tlog[tlog['model']==model].copy()
    scatter = ax.scatter(sub['param_max_depth'], sub['mean_score'],
                         c=sub['sr_trial'], cmap='viridis', s=50, alpha=0.8)
    plt.colorbar(scatter, ax=ax, label='SR of fold scores')
    best = sub.loc[sub['mean_score'].idxmax()]
    ax.scatter(best['param_max_depth'], best['mean_score'],
               marker='*', s=250, color='red', zorder=5, label='Best trial')
    ax.axhline(0.5846, color='grey', ls='--', lw=1, label='Baseline')
    ax.set_xlabel('max_depth')
    ax.set_ylabel('Mean CV accuracy')
    ax.set_title(f'{model}: Tuning Landscape', fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True)
plt.tight_layout()
plt.savefig(f'{FIGURES}/P14_hyperparameter_search.png', **STYLE)
plt.show(); print('Saved P14')

## T10 — Feature Importance Rankings (MDI, MDA, SFI)

In [ ]:
T10 = fimp[['MDI_mean','MDA_mean','SFI_mean','MDI_rank','MDA_rank','SFI_rank','avg_rank','kept']].round(4)
T10 = T10.sort_values('avg_rank')
print('T10: Feature Importance Rankings')
print(T10.to_string())
T10.to_csv(f'{TABLES}/T10_feature_importance.csv')

## T11 — Backtest Statistics (Stage 8)

In [ ]:
T11 = bt_stats.reset_index()
T11.columns = ["Metric", "Value"]
print("T11: Backtest Statistics (5 bps transaction cost)")
print(T11.to_string(index=False))
# already saved in backtest_stats.csv

## T12 — CPCV Path Statistics

In [ ]:
T12 = cpcv_res.copy()
T12["sharpe"] = T12["sharpe"].round(4)
T12 = pd.concat([
    T12,
    pd.DataFrame([{"seed": "Mean", "sharpe": T12["sharpe"].mean().round(4)}]),
    pd.DataFrame([{"seed": "Std",  "sharpe": T12["sharpe"].std().round(4)}]),
], ignore_index=True)
print("T12: CPCV Backtest Path Statistics")
print(T12.to_string(index=False))
T12.to_csv(f"{TABLES}/T12_cpcv_results.csv", index=False)

## T13 — AFML Methods Not Implementable with Daily OHLCV

In [ ]:
T13_data = [
    ('Tick bars',              'Require tick-level timestamps and prices',    'Approximate via daily dollar volume bars'),
    ('Volume bars (exact)',    'Need intraday trade-by-trade volume',          'Cumulate daily volume'),
    ('Tick imbalance bars',   'Need signed tick sequences',                   'Not implementable; theory only'),
    ('VPIN',                  'Requires tick-level BVC classification',        'Approximate from daily O/C direction'),
    ('Kyle Lambda',           'Needs trade-by-trade price impact regression',  'Not implementable'),
    ('Roll spread (accurate)','Needs high-freq covariance of price changes',   'Approximate from daily returns (noisy)'),
    ('Hasbrouck info share',  'Needs multivariate tick data',                  'Not implementable'),
    ('CUSUM on tick returns', 'Designed for intraday returns',                 'Applied to daily returns — valid approx.'),
]
T13 = pd.DataFrame(T13_data, columns=['Method', 'Reason', 'Approximation / Status'])
print('T13: Methods not implementable with daily OHLCV')
print(T13.to_string(index=False))
T13.to_csv(f'{TABLES}/T13_limitations.csv', index=False)

## T14 — Multiprocessing Benchmark

In [ ]:
import time
from src.multiprocess import mp_pandas_obj, _vol_molecule_worker

close = clean["Adj Close"]
index = close.index
print(f"Benchmarking mp_pandas_obj on {len(index)} dates...")

rows = []
baseline = None
for n in [1, 2, 4]:
    t0 = time.perf_counter()
    mp_pandas_obj(_vol_molecule_worker, ("molecule", index), num_threads=n,
                  close=close, span=50)
    elapsed = round(time.perf_counter() - t0, 3)
    if baseline is None:
        baseline = elapsed
    rows.append({"num_threads": n, "elapsed_s": elapsed,
                 "speedup": round(baseline / elapsed, 3) if elapsed > 0 else None})

T14 = pd.DataFrame(rows)
print("T14: Multiprocessing Benchmark (mp_pandas_obj, AFML Snippet 20.5)")
print(T14.to_string(index=False))
T14.to_csv(f"{TABLES}/T14_multiprocessing_benchmark.csv", index=False)

## P22 — SADF (from notebook 12) or generate inline

In [ ]:
if bsadf is not None:
    # Plot inline version (P22 already generated in nb 12 - just reference here)
    valid = bsadf.dropna()
    fig, ax = plt.subplots(figsize=(14, 4), dpi=120)
    ax.plot(valid.index, valid.values, color='darkorange', lw=0.7, label='BSADF')
    ax.axhline(0, color='grey', lw=0.5)
    ax.set_title('BSADF Sequence (see P22 for full chart with CVs)', fontweight='bold')
    ax.set_xlabel('Date'); ax.set_ylabel('BSADF t-stat')
    ax.grid(True)
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    plt.tight_layout()
    plt.show()
    print('BSADF max:', round(float(valid.max()),4))
else:
    print('Run notebook 12 first to generate P22 and nvda_bsadf.parquet')

## P23 & P24 — Entropy and Microstructure (from notebook 13)

In [ ]:
# Recompute for summary display
from src.entropy import rolling_entropy, rolling_lz
from src.microstructure import corwin_schultz_spread, amihud_illiquidity

log_ret = np.log(clean['Adj Close'] / clean['Adj Close'].shift(1))
ent_50  = rolling_entropy(log_ret, window=50)
lz_50   = rolling_lz(log_ret, window=50)
cs_spr  = corwin_schultz_spread(clean['High'], clean['Low'])
amihud  = amihud_illiquidity(clean['Adj Close'], clean['Volume'], window=20)

print('Entropy and microstructure features on full daily series:')
stats_df = pd.DataFrame({
    'Shannon entropy (50d)': ent_50.describe(),
    'LZ complexity (50d)': lz_50.describe(),
    'CS spread': cs_spr.describe(),
    'Amihud illiq.': amihud.describe(),
}).round(6)
print(stats_df.to_string())

## Final Report Outline

In [ ]:
outline = """
AFML NVDA Pipeline — Final Report Outline
==========================================

1. Introduction (AFML Ch. 1)
   - Five challenges of financial ML: non-IID data, engineered labels,
     unequal sample weights, purged CV, overfitting via multiple testing.

2. Data (Ch. 2)
   - NVDA daily OHLCV 2005-2025, 5113 rows. Table T1, Plots P1-P4.
   - Dollar bars (approximate) and CUSUM filter. Tables T2-T3.

3. Labeling and Sample Weights (Ch. 3-4)
   - Triple-barrier method (pt=sl=1.0, 10-day vertical barrier).
   - Concurrency, average uniqueness, return-attribution weights, time decay.
   - Tables T4-T5, Plots P5-P8.

4. Feature Engineering (Ch. 5, 17-19)
   - FFD at d*=0.25 (ADF p=0.012, corr=0.916). Table T6.
   - 17 features: momentum, volatility, volume, microstructure, entropy.
   - Tables T7, Plots P9-P11.

5. Model Training and CV (Ch. 6-7, 9)
   - RF and XGB with PurgedKFold (n=5, embargo=1%).
   - Tables T8-T9, Plots P12-P14.

6. Feature Importance (Ch. 8)
   - MDI, MDA, SFI. Table T10, Plots P15-P17.
   - Pruned: momentum_12_1, bekker_parkinson_vol.

7. Meta-Labeling and Bet Sizing (Ch. 3.6, 10)
   - OOS primary model accuracy: 0.6154. Meta-model F1: 0.5338.
   - Position levels: {-0.2, 0, 0.2, 0.4}. Plot P18.

8. Backtesting (Ch. 11-14)
   - SR=0.291, PSR=0.906, DSR(N=50)=0.168, MaxDD=23.6%. Table T11.
   - CPCV: 5 paths, mean SR=0.626. Table T12, Plots P19-P21.

9. Structural Breaks, Entropy, Microstructure (Ch. 17-19)
   - SADF and bubble detection. Table from nb 12, Plot P22.
   - Rolling entropy, CS spread, Amihud. Plots P23-P24.

10. Limitations
    - 195 samples (low for 15 features). Table T13.
    - Daily OHLCV approximations for tick-level methods.
    - DSR_CV (Stage 5) != true DSR (Stage 8).

11. Performance Optimisation (Ch. 20)
    - mp_pandas_obj: multiprocessing wrapper. Table T14.

12. Conclusion
"""
print(outline)